# Assistants 

[Assistants](https://docs.langchain.com/langsmith/assistants) give developers a quick and easy way to modify and version agents for experimentation.

## Supplying configuration to the graph

Our `task_maistro` graph is already set up to use assistants!

It has a `configuration.py` file defined and loaded in the graph.

We access configurable fields (`user_id`, `todo_category`, `task_maistro_role`) inside the graph nodes.

## Creating assistants 

Now, what is a practical use case for assistants with the `task_maistro` app that we've been building?

For me, it's the ability to have separate ToDo lists for different categories of tasks. 

For example, I want one assistant for my personal tasks and another for my work tasks.

These are easily configurable using the `todo_category` and `task_maistro_role` configurable fields.

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [1]:
%%capture --no-stderr
%pip install -U langgraph_sdk

This is the default assistant that we created when we deployed the graph.

In [4]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### Personal assistant

This is the personal assistant that I'll use to manage my personal tasks.

In [7]:
personal_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": {"todo_category": "personal"}}
)
print(personal_assistant)

{'assistant_id': 'fefd062e-a34e-40ad-b319-690a6006b772', 'graph_id': 'task_maistro', 'created_at': '2025-11-24T07:22:35.127685+00:00', 'updated_at': '2025-11-24T07:22:35.127685+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'metadata': {}, 'version': 1, 'name': 'Untitled', 'description': None, 'context': {'todo_category': 'personal'}}


Let's update this assistant to include my `user_id` for convenience,  [creating a new version of it](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant). 

In [8]:
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

configurations = {"todo_category": "personal", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

{'assistant_id': 'fefd062e-a34e-40ad-b319-690a6006b772', 'graph_id': 'task_maistro', 'created_at': '2025-11-24T07:22:46.953185+00:00', 'updated_at': '2025-11-24T07:22:46.953185+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'personal', 'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging

### Work assistant

Now, let's create a work assistant. I'll use this for my work tasks.

In [5]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {"todo_category": "work", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": configurations}
)
print(work_assistant)

{'assistant_id': '4b9de9bd-95ff-477f-8cd0-dee4575f4eed', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.914775+00:00', 'updated_at': '2025-07-31T18:33:39.914775+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'work', 'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 day

## Using assistants 

Assistants will be saved to `Postgres` in our deployment.  

This allows us to easily search <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [search](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) for assistants with the SDK.

In [9]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

{'assistant_id': 'fefd062e-a34e-40ad-b319-690a6006b772', 'version': 2, 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'personal', 'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging and helpful, never judgmental. \n\nWhen tasks are missing deadlines, respond with something like "I notice [task]

We can manage them easily with the SDK. For example, we can delete assistants that we're no longer using.  
> The syntax in the video is slightly off. The updated code below creates a spare assistant and then deletes it. 

In [10]:
# create a temporary assitant
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")
    
# delete our temporary assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': 60e10126-1f63-4ea4-a113-98be29fab740}
before delete: {'assistant_id': fefd062e-a34e-40ad-b319-690a6006b772}
before delete: {'assistant_id': 3d5b5d51-7a6a-41de-8a77-2aa601497be7}
before delete: {'assistant_id': ea4ebafa-a81d-5063-a5fa-67c755d98a21}

after delete: {'assistant_id': 60e10126-1f63-4ea4-a113-98be29fab740 }
after delete: {'assistant_id': fefd062e-a34e-40ad-b319-690a6006b772 }
after delete: {'assistant_id': 3d5b5d51-7a6a-41de-8a77-2aa601497be7 }


Let's set the assistant IDs for the `personal` and `work` assistants that I'll work with.

In [12]:
assistants

[{'assistant_id': '60e10126-1f63-4ea4-a113-98be29fab740',
  'graph_id': 'task_maistro',
  'created_at': '2025-11-24T07:27:59.666074+00:00',
  'updated_at': '2025-11-24T07:27:59.666074+00:00',
  'config': {'configurable': {'user_id': 'lance',
    'todo_category': 'personal',
    'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style shou

In [11]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

In [14]:
work_assistant_id, personal_assistant_id

('60e10126-1f63-4ea4-a113-98be29fab740',
 'fefd062e-a34e-40ad-b319-690a6006b772')

### Work assistant

Let's add some ToDos for my work assistant.

In [15]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (cb14bc80-87ae-43d7-af4e-fb2d374c3a7d)
 Call ID: cb14bc80-87ae-43d7-af4e-fb2d374c3a7d
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'time_to_complete': None, 'deadline': '2025-11-24', 'task': 'Re-film Module 6, lesson 5'}

New ToDo created:
Content: {'time_to_complete': None, 'deadline': '2025-12-01', 'task': 'Update audioUX'}
================================== Ai Message ==================================

I've updated your ToDo list with "Re-film Module 6, lesson 5" due today and "Update audioUX" due next Monday. Let me know if you need anything else!


In [16]:
user_input = "Create another ToDo: Finalize set of report generation tutorials."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create another ToDo: Finalize set of report generation tutorials.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (20acd884-6980-424b-a422-817180276562)
 Call ID: 20acd884-6980-424b-a422-817180276562
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'time_to_complete': None, 'task': 'Finalize set of report generation tutorials'}
================================== Ai Message ==================================

I've added "Finalize set of report generation tutorials" to your ToDo list! I notice it doesn't have a deadline yet. Would you like to add one to help us track it better?


The assistant uses it's instructions to push back with task creation! 

It asks me to specify a deadline :) 

In [17]:
user_input = "OK, for this task let's get it done by next Tuesday."
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

OK, for this task let's get it done by next Tuesday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (ef4c1ff8-9a5e-4188-a3ff-e09326d5e940)
 Call ID: ef4c1ff8-9a5e-4188-a3ff-e09326d5e940
  Args:
    update_type: todo
================================= Tool Message =================================

Document c41a8eea-ba82-47db-b551-7ddb376a6f84 updated:
Plan: The user wants to set the deadline for the "Finalize set of report generation tutorials" task to next Tuesday. Today is 2025-11-24, so next Tuesday is 2025-12-02. I need to update the "deadline" field of the ToDo item with the ID c41a8eea-ba82-47db-b551-7ddb376a6f84 to "2025-12-02T00:00:00".
Added content: None

Document c41a8eea-ba82-47db-b551-7ddb376a6f84 updated:
Plan: The user wants to set the deadline for the "Finalize set of report generation tutorials" task to next Tuesday. Today is 202

### Personal assistant

Similarly, we can add ToDos for my personal assistant.

In [18]:
user_input = "Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (380ae722-f90e-44b4-bbf3-ad57ff146fc2)
 Call ID: 380ae722-f90e-44b4-bbf3-ad57ff146fc2
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'time_to_complete': 30, 'task': 'Check on swim lessons for the baby this weekend.'}

New ToDo created:
Content: {'time_to_complete': 15, 'task': 'For winter travel, check AmEx points.'}
================================== Ai Message ==================================

I've added "Check on swim lessons for the baby this weekend" and "For winter travel, check AmEx points" to your ToDo list.

I notice "Check on swim lessons for the baby this weekend" doesn't have a deadline ye

In [20]:
user_input = "Give me a todo summary."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Give me a todo summary.
================================== Ai Message ==================================

[{'type': 'text', 'text': "Here's a summary of your current tasks:\n\n**Future Tasks:**\n*   Re-film Module 6, lesson 5 (due 2025-11-24)\n*   Update audioUX (due 2025-12-01)\n\n**Tasks Missing Deadlines:**\n*   For winter travel, check AmEx points. (Estimated time: 15 minutes) - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?\n*   Check on swim lessons for the baby this weekend. (Estimated time: 30 minutes) - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?\n*   Finalize set of report generation tutorials. - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?\n\n**Tasks Missing Time Estimates:**\n*   Finalize set of report generation tutorials. - 

In [23]:
print({
    "type": "text",
    "text": "Here's a summary of your current tasks:\n\n**Future Tasks:**\n*   Re-film Module 6, lesson 5 (due 2025-11-24)\n*   Update audioUX (due 2025-12-01)\n\n**Tasks Missing Deadlines:**\n*   For winter travel, check AmEx points. (Estimated time: 15 minutes) - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?\n*   Check on swim lessons for the baby this weekend. (Estimated time: 30 minutes) - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?\n*   Finalize set of report generation tutorials. - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?\n\n**Tasks Missing Time Estimates:**\n*   Finalize set of report generation tutorials. - This task also seems important but doesn't have a time estimate. Adding one could help you plan your time more effectively!",
    "extras": {
        "signature": "CrAJAdHtim/HgX9kaaQNEWfhs5VJRuqnOVm9HIntPuSdSb98zCOmkf5U1up7Hopuw2viOyQeJqJf7wZxoxsFZ0dCwE7G4UdYgdfOIlMZuGUkpOeiBSg4+li1ROU5a9f1z7pdnzLZJpojWLvT/DC+WT4NiqOo/65CX5gITDo5sBdGyE/K7OAShhg9R9IU2Pvx+gx0ucmAl+ow8NPih6W1KOGVkRFp3iOndCKDhTHAvVhtPHi+tXj6tT3rQHYXr7GpAK5MGbg77i1MfWWYXiTPF5ZHWjF3cA/fqe4xhwQ8ELciOyGVNo5ihV2zWMc08vOVRqsc6H51NtLF3A1kk0TYc0p2qI3B3sAOZpAmzzKpPFOc9zY8W3G3k34OKMOWep5mdtsY/zFHQWq7eXScUmi0MrOC5NyvvKUiDAp16oi7mu8kuZ1I2M98+BQGCkukGkiD8qODlHKhAgHfCwnAknHKZayzT6auVCdK9AjgMsl1RwHwhO+deaiPCVcPVdWgBT2qk/qokRQqJJMow4RCAWypVbDZOZfq48pISGh/y7wF4InLfnTaQ6YNND8/mWTENSOh4ziu925W6N/h9lQksJjYUZ1+SUjuCjQDX73vbQyfMmoJLTdKXF9cdAibQZgt+8xocS3V0L9g2w149fHr3804oGmg9h7I7+YcQTc/niDiMZxm6sW/4tfHmJXAuUxDhOppsfOTd3/kc02rZq8OgToCNf6hlc2InhrnwbjpJpoNDH1/UKSxz4Bv82sH0Are7I+3ewIb8Pha+vCgjpLNpDH6imTfwL7rpzCXqSK/ZI4ZtrZlXLgVsl6eXUjhLyueQLmsTlS//D+PFNVgD+M5/2Og3mJm8Miym+caQjUjWNQk9saeeJK8v4SkfXxdIHJsyfmbd5pmBIrfxXzypdVNFvJN/QXMVNw5SgpHiR+v2g/9GW0wSiDH3wsKrwQ0GZ9KPVI1j6zFcLuf/3TtAIl09EyrQVmDvoFrYvrMNb+YvGT0SoSv7uuKDkmz6HVDBfiHhE+aYbaHvFhJPgpi1Xp7obSa6ALN+K1YEhOYhU+76aRQRmgcvGaqg6VDr7bpAqoF7YXrD2Vpn1r3uoMepuCjMjtZiS+0eT+OEXNDBB5RM5chkNmDG9nZoAeYB2d6fNnMhcOF4HBzgn8NPQ4Ey05Y3QnCXhQPxOFrpYk6YfRIJjNGDNRmllCIfNVLobS6s+h6UVQnA+fEISxTlFiysgDjPe8qo0j/nczHrEx7tHg05XYycHUyzeD08FbnISy7yF7yvg38W5n+I0tvD3sS+US+gh6c1NtCNB5U27NG0MSwbIwMwUAJ/Cpp/Muo5hRQvh8o+/Mo0eMDKwqyffpREnTJ0RSF/zm9ePgVsEEje6bnM/P8lflcgZh/rvCg0x+6HQSpsccXJaJilpiF3/Jj6w30BXqPRICNevgWtDY1k6OOVDqe4PWP6hShcdUNNsL9EU46ggMzVb0N78ubL/lKo6GRRhXibELYZBi/n8AkFeFkKxRKHKnwD47s9EoE2SoFoPMu9CTXN3pkyEI9RyQ+qYB7M0OI9ZcnFXhA0KE5lKOEoNo4noVAxIX4VJ3vDFWYinVOp/Bq34EJ"
    },
}['text'])

Here's a summary of your current tasks:

**Future Tasks:**
*   Re-film Module 6, lesson 5 (due 2025-11-24)
*   Update audioUX (due 2025-12-01)

**Tasks Missing Deadlines:**
*   For winter travel, check AmEx points. (Estimated time: 15 minutes) - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?
*   Check on swim lessons for the baby this weekend. (Estimated time: 30 minutes) - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?
*   Finalize set of report generation tutorials. - I notice this task doesn't have a deadline yet. Would you like to add one to help us track it better?

**Tasks Missing Time Estimates:**
*   Finalize set of report generation tutorials. - This task also seems important but doesn't have a time estimate. Adding one could help you plan your time more effectively!
